# Tutorial 3 — Forward projection and artifacts

**Goal:** see how sinograms are formed and what each acquisition artifact does to them.

**You will learn:** `make_sinogram_pair`, X-ray spectra, the sinogram dictionaries,
`ArtifactConfig` and `inject_sinogram_artifacts`, and the Poisson noise model.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import neutron_xray_sim as nxs

print("DIANA / neutron_xray_sim", nxs.__version__)

## 1. Projection

A projection is the Beer–Lambert law along every ray: $I = I_0 \exp(-\int \mu\,dl)$.
DIANA stores the **optical depth** $\lambda = -\ln(I/I_0)$ (`sino_lam`) and the transmission
$I/I_0$ (`sino_trans`), with shape `(n_angles, n_slices, n_detector)`.

* **X-rays** are polychromatic: the transmitted intensity is a spectrum-weighted sum.
* **Neutrons** are treated as a thermal, monochromatic beam; absorption and scattering
  components are projected separately so the scatter model can use them.

In [ ]:
phantom = nxs.make_phantom("composite", N=48)
xray, neutron = nxs.make_sinogram_pair(phantom, n_angles=90, kVp=120, filter_mm_Al=2.0)
print("keys:", sorted(xray))
print("sinogram shape:", xray["sino_lam"].shape)

In [ ]:
s = phantom.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, sino, t in zip(axes, [xray, neutron], ["X-ray", "neutron"]):
    im = ax.imshow(sino["sino_lam"][:, s, :], aspect="auto", cmap="magma")
    ax.set_title(f"{t} sinogram (central slice)"); ax.set_xlabel("detector pixel")
    ax.set_ylabel("angle index"); plt.colorbar(im, ax=ax, label="optical depth")

### The X-ray spectrum and beam hardening

A tube emits a broad bremsstrahlung spectrum. Low energies are absorbed first, so the beam
"hardens" as it goes through the sample and the measured optical depth grows **less than
linearly** with thickness — the origin of cupping and streaks in X-ray CT.

In [ ]:
E, w = nxs.xray_spectrum(kVp=120, filter_mm_Al=2.0)
t = np.linspace(0, 1.0, 50)                      # iron thickness [cm]
mu = nxs.IRON.mu_x_array(E)
poly = -np.log((w[None, :] * np.exp(-np.outer(t, mu))).sum(1))
mono = nxs.IRON.mu_x_at(np.sum(w * E)) * t        # at the mean energy

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(E, w, width=8); axes[0].set_xlabel("energy [keV]"); axes[0].set_title("120 kVp, 2 mm Al")
axes[1].plot(t, mono, "--", label="monochromatic"); axes[1].plot(t, poly, label="polychromatic")
axes[1].set_xlabel("iron thickness [cm]"); axes[1].set_ylabel("optical depth"); axes[1].legend()

## 2. Artifacts

`ArtifactConfig` has an on/off switch and parameters for each effect. Sinogram artifacts are
applied in the order of the physical detection chain:

1. **scatter** (a blurred halo added to the primary beam) →
2. **detector blur** (PSF) →
3. **Poisson counting noise** →
4. **ring artifacts** (bad detector columns) →
5. optional **beam-hardening correction**.

Volume artifacts (**misalignment** between the two scans, **salt-and-pepper** voxels) are applied
after reconstruction.

In [ ]:
print(nxs.ArtifactConfig.realistic().summary())

Let us apply artifacts one at a time to the neutron sinogram and look at the difference
from the clean data.

In [ ]:
configs = {
    "Poisson noise (I₀=5e3)":  nxs.ArtifactConfig(photon_noise=True, I0_xray=5e3, I0_neutron=5e3),
    "neutron scatter (f=10%)": nxs.ArtifactConfig(neutron_scatter=True, scatter_fraction=0.10),
    "detector PSF (σ=2 px)":   nxs.ArtifactConfig(detector_psf=True, psf_sigma_neutron_pixels=2.0),
    "ring artifacts":          nxs.ArtifactConfig(ring_artifacts=True, n_bad_columns=4,
                                                  ring_amplitude=0.1),
}
fig, axes = plt.subplots(1, len(configs), figsize=(16, 3.8))
for ax, (name, cfg) in zip(axes, configs.items()):
    _, n_art = nxs.inject_sinogram_artifacts(xray, neutron, cfg, rng=np.random.default_rng(0))
    diff = n_art["sino_lam"][:, s, :] - neutron["sino_lam"][:, s, :]
    lim = np.percentile(np.abs(diff), 99.5) or 1
    ax.imshow(diff, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(name, fontsize=10); ax.set_xlabel("detector")
axes[0].set_ylabel("angle"); fig.suptitle("artifact − clean  (neutron optical depth)")

## 3. The noise model

Counting statistics give $\sigma_\lambda \approx \sqrt{e^{\lambda}/I_0}$: noise grows
**exponentially** with attenuation, so the densest path through the sample sets the dose you need.
Let us check the simulation against this prediction.

In [ ]:
I0 = 2e3
cfg = nxs.ArtifactConfig(photon_noise=True, I0_xray=I0, I0_neutron=I0)
lam_clean = neutron["sino_lam"]
samples = [nxs.inject_sinogram_artifacts(xray, neutron, cfg,
                                         rng=np.random.default_rng(k))[1]["sino_lam"]
           for k in range(20)]
measured = np.std(samples, axis=0)

bins = np.linspace(0, lam_clean.max(), 25)
idx = np.digitize(lam_clean, bins)
centres = [lam_clean[idx == i].mean() for i in range(1, len(bins)) if (idx == i).any()]
sigmas = [measured[idx == i].mean() for i in range(1, len(bins)) if (idx == i).any()]
plt.plot(centres, sigmas, "o", label="simulated")
lam = np.linspace(0, lam_clean.max(), 100)
plt.plot(lam, nxs.predicted_sigma_lambda(lam, I0), label=r"$\sqrt{e^\lambda / I_0}$")
plt.xlabel("optical depth λ"); plt.ylabel("σ(λ)"); plt.legend(); plt.title(f"I₀ = {I0:g}")

**Exercise:** the neutron detector of a real beamline often records only $10^2$–$10^4$
counts per pixel, much less than an X-ray tube. Rerun the cell with `I0 = 200`. At what
optical depth does the noise exceed 0.3?